In [29]:
pip install qiskit qiskit-machine-learning qiskit-aer

In [30]:
pip install estimator


In [31]:
!pip install --upgrade qiskit-machine-learning

In [32]:
pip install qiskit qiskit-machine-learning qiskit-aer

In [33]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

# Qiskit components
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import EstimatorV2 # Changed to EstimatorV2
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit.quantum_info import SparsePauliOp

In [40]:
import zipfile

def load_data(path):
    df = pd.read_csv(path)

    # Filter for definitive labels only (Confirmed vs False Positive)
    df = df[df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])]
    df['label'] = df['koi_disposition'].map({
        'CONFIRMED': 1,
        'FALSE POSITIVE': 0
    })

    # Core features relevant to exoplanet detection
    features = [
        'koi_period', 'koi_time0bk', 'koi_impact',
        'koi_duration', 'koi_depth', 'koi_model_snr',
        'koi_prad', 'koi_teq', 'koi_insol',
        'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag'
    ]

    df = df[features + ['label']].dropna()

    X = df[features].values
    y = df['label'].values

    return X, y

def create_vqc(num_input_features, num_qubits_for_dr):
    """
    Creates a Variational Quantum Circuit (VQC) for dimensionality reduction.
    It takes `num_input_features` as input and uses `num_qubits_for_dr` qubits
    to produce `num_qubits_for_dr` output features via observables.
    """
    x = ParameterVector('x', num_input_features) # Input features
    theta = ParameterVector('θ', num_qubits_for_dr) # Trainable parameters for the VQC
    qc = QuantumCircuit(num_qubits_for_dr)

    # Feature embedding: Map input features to Ry gates on qubits.
    # Cycle through qubits if num_input_features > num_qubits_for_dr.
    for i in range(num_input_features):
        qc.ry(x[i], i % num_qubits_for_dr)

    # Entangling layer
    for i in range(num_qubits_for_dr - 1):
        qc.cx(i, i + 1)

    # Variational layer
    for i in range(num_qubits_for_dr):
        qc.ry(theta[i], i)

    return qc, x, theta

def build_qnn(qc, x, theta, num_output_features):

    estimator = EstimatorV2()
    observables = [
        SparsePauliOp.from_list([("I"*i + "Z" + "I"*(qc.num_qubits-1-i), 1)])
        for i in range(num_output_features)
    ]

    qnn = EstimatorQNN(
        circuit=qc,
        observables=observables,
        input_params=x,
        weight_params=theta,
        estimator=estimator
    )
    return qnn, observables

def extract_quantum_features(qnn, X_scaled):
    """Passes classical scaled data through the QNN to get quantum-enhanced features."""
    weights = np.random.rand(qnn.num_weights)

    # qnn.forward will return (num_samples, num_observables)
    output = qnn.forward(X_scaled, weights)

    return np.array(output)

class Classifier(nn.Module):
    """PyTorch CNN for final classification."""
    def __init__(self, input_dim):
        super().__init__()
        # input_dim will be num_qubits_for_dr (e.g., 5)
        self.conv_layers = nn.Sequential(
            # Input: (batch_size, 1, input_dim)
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2), # Reduces sequence length
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )

        with torch.no_grad():
            dummy_input = torch.randn(1, 1, input_dim)
            dummy_output = self.conv_layers(dummy_input)
            flattened_size = dummy_output.view(dummy_output.size(0), -1).size(1)

        self.fc_layers = nn.Sequential(
            nn.Linear(flattened_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

def train_model(X, y):
    """Trains the classical part of the hybrid model."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    model = Classifier(X.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    print("Training Model...")
    for epoch in range(50):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    # Evaluation
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    with torch.no_grad():
        preds = model(X_test_t).numpy()
        preds = (preds > 0.5).astype(int)

    acc = accuracy_score(y_test, preds)
    print(f"\nFinal Test Accuracy: {acc:.4f}")
    return model, acc

def preprocess(X):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled

def run_hybrid_model(X, y):
    print("\n--- Running Hybrid VQC-CNN Model ---")
    num_input_features = X.shape[1]
    num_qubits_for_dr = 5

    X_scaled = preprocess(X) # Only scaling

    print(f"Building Quantum Circuit for DR with {num_input_features} input features and {num_qubits_for_dr} output features...")
    qc, x, theta = create_vqc(num_input_features, num_qubits_for_dr)

    qnn, observables = build_qnn(qc, x, theta, num_qubits_for_dr)

    print("Extracting Quantum Features (VQC-based DR)...")
    X_quantum_reduced = extract_quantum_features(qnn, X_scaled) # This is the X_reduced now

    # The input dimension to the classical classifier needs to be the new reduced dimension
    _, hybrid_acc = train_model(X_quantum_reduced, y) # Pass the quantum-reduced features
    return hybrid_acc

def run_classical_cnn(X, y):
    print("\nRunning Classical CNN Model")
    X_scaled = preprocess(X) # Only scaling
    _, classical_acc = train_model(X_scaled, y)
    return classical_acc

def main():

    zip_path = "exoplanets.csv.zip"
    extracted_csv_path = "exoplanets.csv"

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(".") # Extract to current directory
        print(f"Extracted {extracted_csv_path} from {zip_path}")
    except zipfile.BadZipFile:
        print(f"Error: {zip_path} is not a valid zip file.")
        return
    except FileNotFoundError:
        print(f"Error: {zip_path} not found.")
        return

    print("Loading and preprocessing data...")
    X, y = load_data(path=extracted_csv_path)

    # Remove this line to run on the full dataset
    X, y = X[:500], y[:500]

    hybrid_accuracy = run_hybrid_model(X, y)
    classical_accuracy = run_classical_cnn(X, y)

    print("\n--- Comparison Results ---")
    print(f"Hybrid VQC-CNN Accuracy: {hybrid_accuracy:.4f}")
    print(f"Pure Classical CNN Accuracy: {classical_accuracy:.4f}")


if __name__ == "__main__":
    main()

Extracted exoplanets.csv from exoplanets.csv.zip
Loading and preprocessing data...

--- Running Hybrid VQC-CNN Model ---
Building Quantum Circuit for DR with 13 input features and 5 output features...
Extracting Quantum Features (VQC-based DR)...
Training Model...
Epoch 0, Loss: 0.6901
Epoch 10, Loss: 0.5719
Epoch 20, Loss: 0.5092
Epoch 30, Loss: 0.4549
Epoch 40, Loss: 0.4120

Final Test Accuracy: 0.7600

Running Classical CNN Model
Training Model...
Epoch 0, Loss: 0.7021
Epoch 10, Loss: 0.3511
Epoch 20, Loss: 0.2675
Epoch 30, Loss: 0.2141
Epoch 40, Loss: 0.1765

Final Test Accuracy: 0.9200

--- Comparison Results ---
Hybrid VQC-CNN Accuracy: 0.7600
Pure Classical CNN Accuracy: 0.9200
